# Prepare

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

PROJECT_ROOT = '../../'

## selected data load

In [2]:
csv_path = os.path.join(PROJECT_ROOT, '00_data', '01_interim', 'selected_data_2023.csv')
data_df = pd.read_csv(csv_path, low_memory=False)

#최종 dataframe
final_df = data_df[['pid']]
final_df.head(2)

,pid
0,10002
1,40002


## Helper functions
나중에 test data에 대해서도 적용될 수 있다

### EFA factor 계산

In [3]:
import statsmodels.multivariate.factor as sm_fa

def get_factor_scores(df, n_factor, col_names, rotation_method = 'oblimin', maxiter=1000):

    fa = sm_fa.Factor(df, n_factor=n_factor, method='ml')
    fa_results = fa.fit(maxiter=maxiter)
    fa_results.rotate(rotation_method)

    #factor scores(실제로 사용할 값들)
    factor_scores = fa_results.factor_scoring(method = 'regression')
    scores_df = pd.DataFrame(factor_scores, columns=col_names)
    return scores_df

# 범주형

In [4]:
print(data_df[data_df['p__a03026'].isna()]['is_churned'].value_counts())
print(data_df['p__a03026'].value_counts())

is_churned
1    4
0    2
Name: count, dtype: int64
p__a03026
1.0    1307
2.0     967
Name: count, dtype: int64


In [5]:
# p__c05051 - 0.05 p__l01001 - 0.06 p__a03026 - 0.102(결측치있음)

In [6]:
#최빈값으로 처리
data_df['p__a03026'] = data_df['p__a03026'].fillna(1)
data_df['p__a03026'] = data_df['p__a03026'].map({1:1, 2:0}) # 예 1, 아니오 0
cat_cols = ['p__a03026']
cat_col_names = ['use_infinite_data_plan']

In [7]:
new_df_list = []
for col_name, new_col_name in zip(cat_cols, cat_col_names):
    cat_df = data_df[[col_name]].copy()
    #추가 작업 필요시 수행`
    cat_df.columns = [new_col_name]
    new_df_list.append(cat_df)
final_cat_df = pd.concat(new_df_list, axis=1)
final_df = pd.concat([final_df, final_cat_df], axis=1)
final_df.head(2)

,pid,use_infinite_data_plan
0,10002,1
1,40002,1


# 순위형

### 1 ~ 5 조사 데이터들

In [8]:
pc_cols = ["p__d25040","p__d25041","p__d25042","p__d25045","p__d25047","p__d25048","p__d25049",
    "p__d25052","p__d25053","p__d25056","p__d25057","p__d25058","p__d25060"]

mobile_cols = ["p__d25062","p__d25064","p__d25065","p__d25066","p__d25070","p__d25072","p__d25073",
    "p__d25076","p__d25077","p__d25080","p__d25081","p__d25082","p__d25083","p__d25085"]

privacy_cols = ["p__d23003","p__d23005","p__d23006","p__d23007","p__d23008"]

media_literacy_cols = ["p__n01038","p__n01041","p__n01042","p__n01043","p__n01047"]

newtech_cols = ['p__k020'+str(i).zfill(2) for i in range(1,15)]

pc_factor_names = ['pc_ability', 'internet_ability', 'online_transaction_ability', 'email_ability']
mobile_factor_names = ['mobile_skills', 'mobile_communication_skills', 'mobile_transaction_skills', 'mobile_email_skills']
privacy_factor_name = ['privacy']
media_literacy_factor_name = ['information_evaluation_ability']
newtech_factor_name = ['newtech_perception', 'newtech_impact']

In [9]:
efa_col_groups = [pc_cols, mobile_cols, privacy_cols, media_literacy_cols, newtech_cols]
efa_col_name_groups = [pc_factor_names, mobile_factor_names, privacy_factor_name, media_literacy_factor_name, newtech_factor_name]

In [10]:

for col_names, new_col_names in zip(efa_col_groups, efa_col_name_groups):
    df = data_df[col_names]
    factor_df = get_factor_scores(df, n_factor = len(new_col_names), col_names = new_col_names,)
    final_df = pd.concat([final_df, factor_df], axis =1)

print(final_df.head(2))

     pid  use_infinite_data_plan  pc_ability  internet_ability  \
0  10002                       1   -1.822548          1.183023   
1  40002                       1    0.672746         -0.924118   

   online_transaction_ability  email_ability  mobile_skills  \
0                    0.612023      -0.631893      -2.159055   
1                   -0.860726       0.749981       0.758033   

   mobile_communication_skills  mobile_transaction_skills  \
0                    -0.158017                  -1.735113   
1                    -0.614512                   0.984650   

   mobile_email_skills   privacy  information_evaluation_ability  \
0             1.093058  0.924063                       -0.102966   
1            -0.772916  0.429363                        0.536261   

   newtech_perception  newtech_impact  
0            0.432194        0.023611  
1            1.284843        0.466018  


C:\Users\minhakim\anaconda3\envs\aistudy_env\Lib\site-packages\statsmodels\multivariate\factor.py:417: UserWarning: Fitting did not converge
  warnings.warn("Fitting did not converge")
C:\Users\minhakim\anaconda3\envs\aistudy_env\Lib\site-packages\statsmodels\multivariate\factor_rotation\_gpa_rotation.py:101: RuntimeWarning: divide by zero encountered in log10
  table.append([i_try, f, np.log10(s), al])


In [11]:
final_df = final_df.drop(['mobile_communication_skills', 'mobile_email_skills'], axis = 1)

### 그 외

In [12]:
data_df['p__c01002'] = data_df['p__c01002'].fillna(6) # 중앙값, 평균, 최빈값 모두 동일
other_rank_cols = ['p__age', 'p__c01002']
other_rank_cols_names = ['age_group', 'mobile_charge_monthly']

In [13]:
new_df_list = []
for col_name, new_col_name in zip(other_rank_cols,other_rank_cols_names):
    rank_df = data_df[[col_name]]
    #추가 작업 필요시 수행`
    rank_df.columns = [new_col_name]
    new_df_list.append(rank_df)
final_rank_df = pd.concat(new_df_list, axis=1)
final_df = pd.concat([final_df, final_rank_df], axis = 1)
final_df.head(2)

,pid,use_infinite_data_plan,pc_ability,internet_ability,online_transaction_ability,email_ability,mobile_skills,mobile_transaction_skills,privacy,information_evaluation_ability,newtech_perception,newtech_impact,age_group,mobile_charge_monthly
0,10002,1,-1.822548,1.183023,0.612023,-0.631893,-2.159055,-1.735113,0.924063,-0.102966,0.432194,0.023611,6,5.0
1,40002,1,0.672746,-0.924118,-0.860726,0.749981,0.758033,0.984650,0.429363,0.536261,1.284843,0.466018,5,4.0


# 연속형

In [14]:
data_df['p__a03038'] = data_df['p__a03038'].fillna(data_df['p__a03038'].mean())
num_cols = ['p__a03038', 'p__d26039', 'p__d26043']
num_col_names = ['smartphone_usage_month', 'ott_usage_weekday', 'ott_usage_weekend']

In [15]:
new_df_list = []
for col_name, new_col_name in zip(num_cols,num_col_names):
    num_df = data_df[[col_name]]
    #추가 작업 필요시 수행`
    num_df.columns = [new_col_name]
    new_df_list.append(num_df)
final_num_df = pd.concat(new_df_list, axis=1)
final_df = pd.concat([final_df, final_num_df], axis = 1)
final_df.head(2)

,pid,use_infinite_data_plan,pc_ability,internet_ability,online_transaction_ability,email_ability,mobile_skills,mobile_transaction_skills,privacy,information_evaluation_ability,newtech_perception,newtech_impact,age_group,mobile_charge_monthly,smartphone_usage_month,ott_usage_weekday,ott_usage_weekend
0,10002,1,-1.822548,1.183023,0.612023,-0.631893,-2.159055,-1.735113,0.924063,-0.102966,0.432194,0.023611,6,5.0,48.0,190.0,190.0
1,40002,1,0.672746,-0.924118,-0.860726,0.749981,0.758033,0.984650,0.429363,0.536261,1.284843,0.466018,5,4.0,3.0,120.0,120.0


# 저장

In [16]:
final_df = pd.concat([final_df, data_df['is_churned']], axis = 1)
save_path = os.path.join(PROJECT_ROOT, '00_data', '01_interim', 'train.csv')
final_df.to_csv(save_path, index = False)